<a href="https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**ML task type:** Ranking. A score for each page is simply the tool used to put pages in order.

**Why ranking, not classification:** A content team does not only need to know whether a page looks weak. It has limited review capacity, so it needs to know which K pages to investigate first. A yes/no classification would not tell the team which pages deserve attention first.

**Output and action:** An ordered list of pages, each accompanied by the main signals that influenced its position. An editor uses this list to decide which pages to investigate. It is decision-support, not evidence that editing a page will help.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: review_candidate — a provisional binary proxy indicating whether a page should be prioritised for content review.

Definition:
- The page had at least a 30% decline in impressions between the previous and most recent 30-day periods.
- Its previous-period impressions were at or above the 75th percentile of impressions for its own client.
Why this proxy: A decline alone could prioritise pages with very little traffic. Combining decline with client-relative exposure helps identify pages where investigating a decline may be more worthwhile.

Limitations: This is a historical proxy, not a verified editorial label. A decline does not prove that a page needs updating or that an update will improve its performance. Pages with zero previous-period impressions are excluded because their percentage change cannot be meaningfully calculated.

In [16]:
!git clone https://github.com/neelamobaid/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 124 (delta 37), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 10.02 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [17]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [18]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [19]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [20]:
# Compare impressions in the previous and most recent 30-day periods

print("Pages with previous-period impressions:",
      (df["impressions_prev_30d"] > 0).sum())

print("Pages with lower recent impressions:",
      (df["impressions_last_30d"] < df["impressions_prev_30d"]).sum())

print("Pages with higher recent impressions:",
      (df["impressions_last_30d"] > df["impressions_prev_30d"]).sum())

print("Pages with unchanged impressions:",
      (df["impressions_last_30d"] == df["impressions_prev_30d"]).sum())

Pages with previous-period impressions: 26612
Pages with lower recent impressions: 19716
Pages with higher recent impressions: 8690
Pages with unchanged impressions: 1594


In [21]:
# Only compare pages that had impressions in the previous period
eligible = df[df["impressions_prev_30d"] > 0].copy()

eligible["impression_change_pct"] = (
    (eligible["impressions_last_30d"] - eligible["impressions_prev_30d"])
    / eligible["impressions_prev_30d"]
) * 100

print("Eligible pages:", len(eligible))
print("\nPercentage change distribution:")
print(eligible["impression_change_pct"].describe())

print("\nPages with significant impression declines:")

for threshold in [20, 30, 50]:
    count = (eligible["impression_change_pct"] <= -threshold).sum()
    print(f"Decline of at least {threshold}%: {count} pages")

Eligible pages: 26612

Percentage change distribution:
count    26612.000000
mean        -4.785741
std        473.861561
min       -100.000000
25%        -62.632201
50%        -33.464585
75%          0.000000
max      44900.000000
Name: impression_change_pct, dtype: float64

Pages with significant impression declines:
Decline of at least 20%: 16305 pages
Decline of at least 30%: 14129 pages
Decline of at least 50%: 9642 pages


In [22]:
# Inspect previous-period impressions

print("Previous-period impression statistics:")
print(eligible["impressions_prev_30d"].describe())

print("\nPages with fewer than 100 previous-period impressions:",
      (eligible["impressions_prev_30d"] < 100).sum())

print("Pages with at least 100 previous-period impressions:",
      (eligible["impressions_prev_30d"] >= 100).sum())

Previous-period impression statistics:
count     26612.000000
mean       2010.083985
std        6495.193557
min           1.000000
25%          55.750000
50%         305.500000
75%        1397.000000
max      218786.000000
Name: impressions_prev_30d, dtype: float64

Pages with fewer than 100 previous-period impressions: 8602
Pages with at least 100 previous-period impressions: 18010


In [23]:
# Find the top 25% exposure threshold within each client

eligible["client_exposure_threshold"] = (
    eligible.groupby("client_id")["impressions_prev_30d"]
    .transform(lambda s: s.quantile(0.75))
)

# Define the provisional review-candidate proxy

eligible["review_candidate"] = (
    (eligible["impression_change_pct"] <= -30) &
    (eligible["impressions_prev_30d"] >= eligible["client_exposure_threshold"])
).astype(int)

print("Eligible pages:", len(eligible))
print("Review candidates:", eligible["review_candidate"].sum())
print("Candidate share:", f"{eligible['review_candidate'].mean():.1%}")

Eligible pages: 26612
Review candidates: 3200
Candidate share: 12.0%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary success metric: Precision@50

Precision@50 measures how many of the 50 highest-ranked pages are proxy-positive.

Why this metric: The content team has limited review capacity. We want the ranking to place as many potentially important review candidates as possible near the top, rather than simply achieving good overall accuracy across all pages.

Business interpretation: A higher Precision@50 means less of the team's limited review time is spent investigating pages that do not meet our proxy definition.

Limitation: The metric evaluates agreement with our historical proxy, not whether reviewing or updating a page will improve its performance.

In [24]:
# Calculate the random-ranking baseline

random_baseline = eligible["review_candidate"].mean()

print("Review capacity (K): 50")
print("Random-ranking Precision@50 baseline:",
      f"{random_baseline:.1%}")

Review capacity (K): 50
Random-ranking Precision@50 baseline: 12.0%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: One individual content page belonging to a client.

Dataset structure: The dataset contains 30,000 rows, 30,000 unique content IDs, and 32 unique clients. Each row represents one page, with its associated content characteristics and performance measurements.

Why this unit: The content team makes review decisions at the individual page level, so each page needs its own score and position in the review queue.

Identifiers: content_id and client_id identify pages and their respective clients. They will be retained for tracking and grouping, but not used as predictive features.

In [25]:
print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

print("\nExample rows:")
print(df[["content_id", "client_id", "content_type"]].head())

Total rows: 30000
Unique content IDs: 30000
Unique clients: 32

Example rows:
             content_id          client_id     content_type
0  content_304f48230142  client_f369cb89fc  keyword article
1  content_a1fb4e703a9e  client_4e07408562  keyword article
2  content_9aa793d4d895  client_7f2253d7e2  keyword article
3  content_331d6c4de07b  client_19581e27de  keyword article
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article


In [26]:
print("\nLane dataset shape:", eligible.shape)

print("\nExample pages:")
print(eligible[[
    "content_id",
    "client_id",
    "content_type",
    "impression_change_pct",
    "review_candidate"
]].head())


Lane dataset shape: (26612, 47)

Example pages:
             content_id          client_id     content_type  \
0  content_304f48230142  client_f369cb89fc  keyword article   
1  content_a1fb4e703a9e  client_4e07408562  keyword article   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article   
3  content_331d6c4de07b  client_19581e27de  keyword article   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article   

   impression_change_pct  review_candidate  
0             -41.438703                 1  
1             -57.717667                 1  
2             -60.880276                 1  
3             -13.789824                 0  
4             -34.733416                 1  


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why machine learning: A fixed rule can identify pages using one or two predefined conditions, but it may overlook combinations of signals. Machine learning could learn relationships between multiple page characteristics, such as content age, search performance, content type, and engagement. We will compare its ranking against a simple rule-based baseline to see whether it actually improves the review queue.

Comparison: We can compare a simple rule-based ranking with an ML-based ranking using Precision@50. This will help determine whether ML actually improves the quality of the review queue.

Limitation: ML is only useful if it performs better than a simple baseline. It does not prove that updating a selected page will improve its performance.

In [27]:
# Compare the proxy-positive rate with our review capacity

print("Total eligible pages:", len(eligible))
print("Proxy-positive pages:", eligible["review_candidate"].sum())
print("Proxy-positive rate:", f"{eligible['review_candidate'].mean():.1%}")
print("Pages available for review:", 50)

Total eligible pages: 26612
Proxy-positive pages: 3200
Proxy-positive rate: 12.0%
Pages available for review: 50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.